In [ ]:
%%capture
!pip install dash
!pip install dash-mantine-components
!pip install dash-bootstrap-components

In [ ]:
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
import pandas as pd
import re
import plotly.express as px
import geopandas as gpd
from collections import Counter
import numpy as np
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import warnings
warnings.filterwarnings("ignore")
from urllib.request import urlopen
import json
from PIL import Image
from wordcloud import WordCloud, STOPWORDS
import io
import base64

Mounted at /content/drive


In [ ]:
df = pd.read_csv("bigfoot.csv")
df["Season"] = df["Season"].replace("Unknown", np.nan)
df["Date"] = pd.to_datetime({
    "year": df["Year"],
    "month": pd.to_datetime(df["Month"], format="%B").dt.month,
    "day": df["Date"]
}, errors='coerce')

In [ ]:
# полигоны для построения карты counties
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

In [ ]:
def detect_activity(text):
    text = str(text).lower()
    activity_map = {
        'Outdoor activities': ['hiking', 'driving', 'walking', 'biking', 'jogging', 'skiing',
                           'hike', 'camp', 'fish', 'trail', 'hunt', 'outdoors', 'mountain'],
        'Passive rest': ['watching tv', 'eating', 'cleaning', 'cooking', 'sleep', 'rest', 'home', 'bed'],
        'Social situation': ['bar', 'family', 'celebration', 'friend', 'group', 'party', 'social'],
        'Work': ['construction', 'delivering', 'patrolling', 'shift', 'work', 'job', 'logging'],
    }
    for category, keywords in activity_map.items():
        if any(k in text for k in keywords):
            return category
    return 'Other'

In [ ]:
feature_keywords = {
    'Appearance': [r'\btall\b', r'8-10\s?feet', r'hairy', r'brown', r'black', r'muscular', r'\bbig\b', r'red eyes?'],
    'Behaviour': [r'running', r'walking', r'hiding', r'throwing', r'eating', r'crossing', r'following'],
    'Sounds': [r'growl', r'scream', r'howl', r'roar'],
    'Smell': [r'smell', r'stench', r'odor'],
}

def detect_features(text):
    categories = set()
    for category, patterns in feature_keywords.items():
        for pattern in patterns:
            if re.search(pattern, text):
                categories.add(category)
    return list(categories)

In [ ]:
custom_stopwords = set(STOPWORDS)
custom_stopwords.update([
    'the', 'and', 'was', 'with', 'that', 'have', 'from', 'for', 'were',
    'they', 'this', 'but', 'just', 'had', 'been', 'what', 'him', 'her',
    'his', 'she', 'said', 'then', 'about', 'out', 'could', 'would', 'there',
    'their', 'some', 'when', 'over', 'into', 'one', 'them', 'because'
])


def generate_wordcloud_image(text):
    wordcloud = WordCloud(
        width=800,
        height=420,
        background_color='#1e1e2f',
        stopwords=custom_stopwords,
        colormap='magma'
    ).generate(text)

    image = wordcloud.to_image()
    buffer = io.BytesIO()
    image.save(buffer, format='PNG')
    buffer.seek(0)
    encoded = base64.b64encode(buffer.read()).decode()
    return f'data:image/png;base64,{encoded}'

In [ ]:
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])
server = app.server

external_styles = {
    'backgroundColor': '#1e1e2f',
    'color': '#ffffff',
    'fontFamily': 'Arial, sans-serif',
}
app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("Bigfoot Report Dashboard", className='text-center text-danger mb-4'), width=9)
    ]),
    dbc.Row([
        dbc.Col(dcc.RangeSlider(
            id='year-slider',
            min=df['Year'].min(),
            max=df['Year'].max(),
            step=1,
            marks={i: str(i) for i in range(df['Year'].min(), df['Year'].max()+1, 10)},
            value=[1990, 2020],
            tooltip={"placement": "bottom", "always_visible": True}
        ), width=10)
    ], className='mb-4'),

    dbc.Row([
        # Левая колонка (1/3 ширины)
        dbc.Col([
            dbc.Row([
                dbc.Col([
                    html.H5("Total reports", className="text-center"),
                    html.H2(id='total-indicator', className="text-center text-danger", style={'fontSize': '3em'})
                ])
            ]),
            dbc.Row([
                dbc.Col(dcc.Graph(id='yearly'))
            ])
        ], width=4),

        # Правая колонка (2/3 ширины)
        dbc.Col([
            dcc.Graph(id='map', style={"height": "100%"})
        ], width=6)
    ], className='mb-4'),

    dbc.Row([
        dbc.Col(dcc.Graph(id='season'), width=3),
        dbc.Col(dcc.Graph(id='weekday'), width=3),
        dbc.Col(dcc.Graph(id='context'), width=4)
    ], className='mb-4'),
    dbc.Row([
        dbc.Col(dcc.Graph(id='features'), width=5),
        # dbc.Col([
        #     html.H4("Reports Wordcloud", className="text-center"),
        #     html.Img(id='wordcloud-img', style={'width': '100%', 'borderRadius': '10px'})
        # ], width=7)
        dbc.Col([
        html.H5("Reports Wordcloud", className="text-center text-light"),
        html.Img(id='wordcloud-img', style={
            'width': '100%',
            'maxWidth': '100%',
            'height': '420px',
            # 'objectFit': 'cover',
        })
    ], width=5)
    ], className="mb-4")
], fluid=True)

@app.callback(
    Output('map', 'figure'),
    Output('yearly', 'figure'),
    Output('season', 'figure'),
    Output('weekday', 'figure'),
    Output('context', 'figure'),
    Output('features', 'figure'),
    Output('wordcloud-img', 'src'),
    Output('total-indicator', 'children'),
    Input('year-slider', 'value')
)
def update_dashboard(year_range):
    """"Обновление дашборда"""
    dff = df[(df['Year'] >= year_range[0]) & (df['Year'] <= year_range[1])]

    county_count = dff["County"].value_counts().reset_index()
    county_count.columns = ['County', 'count']

    # Универсальные настройки тёмной темы
    dark_layout = dict(
        paper_bgcolor=external_styles['backgroundColor'],
        plot_bgcolor=external_styles['backgroundColor'],
        font_color=external_styles['color'],
        font=dict(family=external_styles['fontFamily']),
        xaxis=dict(color='white'),
        yaxis=dict(color='white'),
        legend=dict(font=dict(color='white')),
        title_font=dict(color='white')
    )

    # Карта
    fig_map = px.choropleth(
        county_count, geojson=counties, locations='County',
        featureidkey="properties.NAME",
        color='count', color_continuous_scale="Reds",
        range_color=(0, county_count["count"].max()),
        scope="usa",
        title="Map of encounters"
    )
    fig_map.update_layout(**dark_layout)
    fig_map.update_geos(
        bgcolor=external_styles['backgroundColor'],

    )

    # По годам
    yearly = dff.groupby('Year').size().reset_index(name='Count')
    fig_year = px.line(
        yearly, x='Year', y='Count', markers=True,
        title="By year", color_discrete_sequence=["orange"]
    )
    fig_year.update_layout(**dark_layout)

    # По сезонам
    fig_season = px.bar(
        dff['Season'].value_counts().rename_axis('Season').reset_index(name='Count'),
        x='Season', y='Count', title="By season",
        color_discrete_sequence=["#fcbf49"],
        category_orders={"Season": ["Winter", "Spring", "Summer", "Fall"]}
    )
    fig_season.update_layout(**dark_layout)

    # По дням недели
    fig_weekday = px.bar(
        dff["Date"].dt.day_name().value_counts().rename_axis('Weekday').reset_index(name='Count'),
        x='Weekday', y='Count', title="By weekday",
        category_orders={"Weekday": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]},
        color_discrete_sequence=["#ffd60a"]
    )
    fig_weekday.update_layout(**dark_layout)

    # Обстоятельства встречи
    fig_context = px.pie(
        dff["Observed"].apply(detect_activity).value_counts().rename_axis('Context').reset_index(name='Count'),
        values='Count', names='Context', title="Context of encounters",
        color_discrete_sequence=px.colors.sequential.Plasma
    )
    fig_context.update_layout(**dark_layout)

    # Описание из наблюдений
    all_features = sum(dff['Follow-Up Report'].fillna('').str.lower().apply(detect_features), [])
    feature_counts = Counter(all_features)
    features_df = pd.DataFrame.from_dict(feature_counts, orient='index', columns=['Count']).reset_index()
    features_df.columns = ['Feature', 'Count']

    fig_features = px.pie(
        features_df, names='Feature', values='Count',
        title='Description about Bigfoot',
        color_discrete_sequence=px.colors.sequential.RdBu
    )
    fig_features.update_traces(textinfo='percent+label')
    fig_features.update_layout(**dark_layout)

    # WordCloud
    text = " ".join(dff['Observed'].dropna().astype(str).tolist()).lower()
    wordcloud_img = generate_wordcloud_image(text)

    # Текст "Всего случаев"
    total_text = f"{len(dff):,}"

    return fig_map, fig_year, fig_season, fig_weekday, fig_context, fig_features, wordcloud_img, total_text

app.run(debug=True)

<IPython.core.display.Javascript object>